In [1]:
# Load libraries
suppressMessages(library(MAST))
suppressMessages(library(lme4))
suppressMessages(library(data.table))
suppressMessages(require("Matrix"))
library("reticulate")

Warning message:
“package ‘MAST’ was built under R version 4.3.2”
Warning message:
“package ‘SingleCellExperiment’ was built under R version 4.3.2”
Warning message:
“package ‘SummarizedExperiment’ was built under R version 4.3.2”
Warning message:
“package ‘MatrixGenerics’ was built under R version 4.3.2”
Warning message:
“package ‘GenomicRanges’ was built under R version 4.3.2”
Warning message:
“package ‘BiocGenerics’ was built under R version 4.3.2”
Warning message:
“package ‘S4Vectors’ was built under R version 4.3.2”
Warning message:
“package ‘IRanges’ was built under R version 4.3.2”
Warning message:
“package ‘GenomeInfoDb’ was built under R version 4.3.2”
Warning message:
“package ‘Biobase’ was built under R version 4.3.2”


In [2]:
# Use the path to your Python executable in the virtual environment
use_python("/lila/home/forsythb/.virtualenvs/r-reticulate/bin/")

# Look at where Python is located
py_config()

python:         /lila/home/forsythb/.virtualenvs/r-reticulate/bin/python
libpython:      /home/forsythb/anaconda3/lib/libpython3.8.so
pythonhome:     /lila/home/forsythb/.virtualenvs/r-reticulate:/lila/home/forsythb/.virtualenvs/r-reticulate
version:        3.8.18 (default, Sep 11 2023, 13:40:15)  [GCC 11.2.0]
numpy:          /lila/home/forsythb/.virtualenvs/r-reticulate/lib/python3.8/site-packages/numpy
numpy_version:  1.24.4

NOTE: Python version was forced by use_python() function

In [3]:
# Read the leiden value from command-line argument
args <- commandArgs(trailingOnly = TRUE)
leiden_value <- as.numeric(args[1])

Warning message in eval(expr, envir, enclos):
“NAs introduced by coercion”


In [5]:
# Set leiden value manually
leiden_value <- "6"  # Change this to the desired leiden value

# Import scanpy
sc <- import("scanpy")

In [6]:
# Read in the adata for primary tumor site
adata <- sc$read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/postprocess_adata/adata.combined.postprocess.leiden.h5ad')

In [28]:
# Load libraries
suppressMessages(library(MAST))
suppressMessages(library(lme4))
suppressMessages(library(data.table))
suppressMessages(require("Matrix"))
suppressMessages(library(reticulate))

args <- commandArgs(trailingOnly = TRUE)

In [29]:
# Use the path to your Python executable in the virtual environment
use_python("/lila/home/forsythb/.virtualenvs/r-reticulate/bin/")

# Look at where Python is located
py_config()

# Import scanpy
sc <- import("scanpy")

# Concatenated adata
adata = sc$read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/postprocess_adata/postprocess_adata.020624/adata.combined.postprocess.h5ad')

python:         /lila/home/forsythb/.virtualenvs/r-reticulate/bin/python
libpython:      /home/forsythb/anaconda3/lib/libpython3.8.so
pythonhome:     /lila/home/forsythb/.virtualenvs/r-reticulate:/lila/home/forsythb/.virtualenvs/r-reticulate
version:        3.8.18 (default, Sep 11 2023, 13:40:15)  [GCC 11.2.0]
numpy:          /lila/home/forsythb/.virtualenvs/r-reticulate/lib/python3.8/site-packages/numpy
numpy_version:  1.24.4

NOTE: Python version was forced by use_python() function

In [30]:
# Extract necessary information from your AnnData object
expression_matrix <- as.matrix(adata$X)

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 12.0 GiB”


In [34]:
# Define a single cell experiment, which is required by MAST
sce = SingleCellExperiment(list(expression_matrix))
sce

class: SingleCellExperiment 
dim: 50906 31586 
metadata(0):
assays(1): ''
rownames(50906): KG146Li_BASE_shZFP36L2_4_GTCGAATTCGGACGTC-1
  KG146Li_HISC_shCtrl_GGATCTAAGGTTATAG-1 ...
  146P_dedifferentiation_shZFP36L2_4_TCGGGTGCAAACGGCA-1
  146P_BASE_shZFP36L2_4_ACTGCAATCAGTGTTG-1
rowData names(0):
colnames(31586): A1BG A1BG-AS1 ... ZYX ZZEF1
colData names(0):
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [41]:
cdr2 <-colSums(assay(sce)>0)
colData(sce)$cngeneson <- scale(cdr2)

In [43]:
colnames(colData(sce))

[1] "cngeneson"

In [33]:
# Keep genes that are expressed in more than 10% of all cells
sce <- sce[freq(sce)>0.1,]
print("Dimensions after subsetting:")
print(dim(sca))

ERROR: Error in (function (cond) : error in evaluating the argument 'i' in selecting a method for function '[': is(sc, "SingleCellAssay") is not TRUE


In [11]:
find_de_MAST_RE <- function(adata_){
    # create a MAST object
    sca <- SceToSingleCellAssay(adata_, class = "SingleCellAssay")
    print("Dimensions before subsetting:")
    print(dim(sca))
    print("")
    # keep genes that are expressed in more than 10% of all cells
    sca <- sca[freq(sca)>0.1,]
    print("Dimensions after subsetting:")
    print(dim(sca))
    print("")
    # add a column to the data which contains scaled number of genes that are expressed in each cell
    cdr2 <- colSums(assay(sca)>0)
    colData(sca)$ngeneson <- scale(cdr2)
    # store the columns that we are interested in as factors
    label <- factor(colData(sca)$label)
    # set the reference level
    label <- relevel(label,"ctrl")
    colData(sca)$label <- label
    celltype <- factor(colData(sca)$cell_type)
    colData(sca)$celltype <- celltype
    # same for donors (which we need to model random effects)
    replicate <- factor(colData(sca)$replicate)
    colData(sca)$replicate <- replicate
    # create a group per condition-celltype combination
    colData(sca)$group <- paste0(colData(adata_)$label, ".", colData(adata_)$cell_type)
    colData(sca)$group <- factor(colData(sca)$group)
    # define and fit the model
    zlmCond <- zlm(formula = ~ngeneson + group + (1 | replicate), 
                   sca=sca, 
                   method='glmer', 
                   ebayes=F, 
                   strictConvergence=F,
                   fitArgsD=list(nAGQ = 0)) # to speed up calculations
    
    # perform likelihood-ratio test for the condition that we are interested in    
    summaryCond <- summary(zlmCond, doLRT='groupstim.CD14_Monocytes')
    # get the table with log-fold changes and p-values
    summaryDt <- summaryCond$datatable
    result <- merge(summaryDt[contrast=='groupstim.CD14_Monocytes' & component=='H',.(primerid, `Pr(>Chisq)`)], # p-values
                     summaryDt[contrast=='groupstim.CD14_Monocytes' & component=='logFC', .(primerid, coef)],
                     by='primerid') # logFC coefficients
    # MAST uses natural logarithm so we convert the coefficients to log2 base to be comparable to edgeR
    result[,coef:=result[,coef]/log(2)]
    # do multiple testing correction
    result[,FDR:=p.adjust(`Pr(>Chisq)`, 'fdr')]
    result = result[result$FDR<0.01,, drop=F]

    result <- stats::na.omit(as.data.frame(result))
    return(result)
}

In [12]:
result <- find_de_MAST_RE(adata)

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'X' in selecting a method for function 'lapply': unable to find an inherited method for function ‘assays’ for signature ‘"AnnDataR6"’


In [16]:
sca <- SceToSingleCellAssay(adata, class = "SingleCellAssay")

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'X' in selecting a method for function 'lapply': unable to find an inherited method for function ‘assays’ for signature ‘"AnnDataR6"’


In [21]:
# Extract necessary information from your AnnData object
expression_matrix <- as.matrix(adata$X)

Warning message in asMethod(object):
“sparse->dense coercion: allocating vector of size 12.0 GiB”


In [22]:
expression_matrix

,A1BG,A1BG-AS1,A1CF,A2M,A2M-AS1,A2ML1,A2ML1-AS1,A2ML1-AS2,A3GALT2,A4GALT,⋯,ZW10,ZWILCH,ZWINT,ZXDA,ZXDB,ZXDC,ZYG11A,ZYG11B,ZYX,ZZEF1
KG146Li_BASE_shZFP36L2_4_GTCGAATTCGGACGTC-1,0,0,0.000000,0,0,0,0,0,0,0,⋯,0,0,0.000000,0,0,0.000000,0,0.000000,0,0.000000
KG146Li_HISC_shCtrl_GGATCTAAGGTTATAG-1,0,0,0.000000,0,0,0,0,0,0,0,⋯,0,0,0.000000,0,0,1.000000,0,0.000000,0,0.000000
146P_dedifferentiation_shCtrl_CTCTCAGAGGTAGACC-1,0,0,1.000000,0,0,0,0,0,0,0,⋯,0,0,1.000000,0,0,0.000000,0,0.000000,0,0.000000
146P_HISC_shZFP36L2_4_GTGCAGCAGAGAGGGC-1,0,0,0.000000,0,0,0,0,0,0,0,⋯,0,0,0.000000,0,0,0.000000,0,0.000000,0,1.000000
146P_BASE_shZFP36L2_3_TAGAGTCAGAAAGTCT-1,0,0,0.000000,0,0,0,0,0,0,0,⋯,0,0,1.000000,0,0,0.000000,0,0.000000,0,0.000000
146P_BASE_shCTRL_GGGTCACGTGATACTC-1,0,0,2.000000,0,0,0,0,0,0,0,⋯,0,0,0.000000,0,0,0.000000,0,0.000000,0,1.000000
KG146Li_BASE_shZFP36L2_4_TTGTTGTAGTCACTGT-1,0,0,0.000000,0,0,0,0,0,0,0,⋯,0,0,0.000000,0,0,0.000000,0,0.000000,0,0.000000
146P_BASE_shCTRL_GGAGATGAGTTCCAGT-1,0,0,0.000000,0,0,0,0,0,0,0,⋯,0,0,0.000000,0,0,0.000000,0,1.000000,0,0.000000
146P_BASE_shZFP36L2_4_GAGTTTGCAAGACGGT-1,0,0,0.000000,0,0,0,0,0,0,0,⋯,1,0,1.000000,0,0,0.000000,0,0.000000,0,0.000000
146P_BASE_shZFP36L2_3_CTAACCCTCACATCAG-1,0,0,0.000000,0,0,0,0,0,0,0,⋯,0,0,0.000000,0,0,0.000000,0,1.000000,0,0.000000


In [24]:
corownames = 

ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'as.data.frame': could not find function "annotations"


In [19]:
# Convert AnnData to SingleCellExperiment
sce <- SingleCellExperiment(
  assays = list(counts = assays(adata)$X),
  colData = as.data.frame(annotations(adata))
)

# Print summary of the SingleCellExperiment object
summary(sce)

ERROR: Error in (function (classes, fdef, mtable) : unable to find an inherited method for function ‘assays’ for signature ‘"AnnDataR6"’


In [27]:
sce = SingleCellExperiment(expression_matrix)
sce

class: SingleCellExperiment 
dim: 50906 31586 
metadata(0):
assays(1): ''
rownames(50906): KG146Li_BASE_shZFP36L2_4_GTCGAATTCGGACGTC-1
  KG146Li_HISC_shCtrl_GGATCTAAGGTTATAG-1 ...
  146P_dedifferentiation_shZFP36L2_4_TCGGGTGCAAACGGCA-1
  146P_BASE_shZFP36L2_4_ACTGCAATCAGTGTTG-1
rowData names(0):
colnames(31586): A1BG A1BG-AS1 ... ZYX ZZEF1
colData names(0):
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):